# 11. Set-based deterministic uncertainty

The drone from notebook 07 is extended so the battery has two internal parameters (specific energy, efficiency) that are known only up to an uncertainty set. The question is: under the worst-case point of that set, how heavy does the drone become?

The nominal parameters (specific_energy = 2.0 MJ/kg, efficiency = 0.90) have a product of 1.8 MJ/kg of *delivered* energy density, so the nominal solve converges to exactly the canonical drone's 0.5492 kg (notebooks 01/06/07). The uncertainty sets perturb around that shared reference.

Two sets are exercised:

- A `Box`: rectangular ranges on the two parameters, with each range declared in the "more is better" direction so the worst case is the corner where both parameters take their lowest values.
- An `Ellipsoid`: a tilted, correlated set smaller than the box; the worst case lies on its boundary in the direction of badness, not at a corner.

Set-based uncertainty fits MCDP naturally: monotonicity means the worst case is always on the set's boundary, in the direction of badness, and the answer is a single antichain rather than a distribution.


## Imports and modules

In [1]:
from codesign import Box, Ellipsoid, Module, Reals, System, solve

class Battery(Module):
    F = {"capacity": Reals(unit="J")}
    R = {"mass":     Reals(unit="kg")}
    def __init__(self, specific_energy=2.0e6, efficiency=0.9):
        self.specific_energy = specific_energy
        self.efficiency = efficiency
        super().__init__()
    def h(self, f):
        return {"mass": f["capacity"] / (self.specific_energy * self.efficiency)}

class Actuator(Module):
    F = {"lift_force": Reals(unit="N")}
    R = {"power":      Reals(unit="W")}
    def h(self, f):
        return {"power": 10.0 * f["lift_force"] ** 2}

def make_drone(battery):
    sys = System("drone")
    endurance     = sys.provides("endurance",     unit="s")
    extra_payload = sys.provides("extra_payload", unit="kg")
    extra_power   = sys.provides("extra_power",   unit="W")
    total_mass    = sys.requires("total_mass",    unit="kg")
    b = sys.add("battery",  battery)
    a = sys.add("actuator", Actuator())
    b.capacity    >= (a.power + extra_power) * endurance
    a.lift_force  >= 9.81 * (b.mass + extra_payload)
    total_mass    >= b.mass + extra_payload
    return sys.build()

f = {"endurance": 300.0, "extra_payload": 0.5, "extra_power": 5.0}

## Nominal: no uncertainty

In [2]:
drone = make_drone(Battery())
nominal = solve(drone, f)
nominal_mass = list(nominal.antichain.points)[0]["total_mass"]
print(f"nominal total_mass = {nominal_mass:.4f} kg")

nominal total_mass = 0.5492 kg


## Box uncertainty

The Box puts independent ranges on each parameter. Because both are declared "more is better," the worst case is the single corner where specific_energy = 1.7e6 and efficiency = 0.83.


In [3]:
bat = Battery()
# Box: independent ranges on each parameter. The direction-of-badness token
# tells the solver which endpoint is the "worst" - for both parameters here,
# lower is worse since they're declared "more_is_better".
bat.uncertain_set = Box(
    specific_energy=(1.7e6, 2.3e6, "more_is_better"),
    efficiency=(0.83, 0.97, "more_is_better"),
)
drone = make_drone(bat)

# uncertainty=["worst_case"] runs one solve at the worst corner of the box.
r_box = solve(drone, f, uncertainty=["worst_case"])
wc = list(r_box.worst_case.antichain.points)[0]["total_mass"]
print(f"Box worst case: {wc:.4f} kg  (penalty {wc - nominal_mass:+.4f} kg)")

Box worst case: 0.5668 kg  (penalty +0.0176 kg)


## Ellipsoid uncertainty (smaller, correlated set)

The Ellipsoid carves out the implausible corner where both parameters are simultaneously at their extremes. The worst case lies on the curved boundary in the direction of badness, which is closer to the centre than the box's worst-case corner.


In [4]:
bat = Battery()
# Ellipsoid: a tilted set centred at the nominal values. The negative
# off-diagonal cov entry encodes a negative correlation: pessimistic on
# specific_energy makes optimistic on efficiency more likely, and vice
# versa. The worst case is therefore not the joint-pessimistic corner.
bat.uncertain_set = Ellipsoid(
    center={"specific_energy": 2.0e6, "efficiency": 0.9},
    cov=[
        [1.0e10, -2.0e3],     # variance of specific_energy, covariance
        [-2.0e3,  2.5e-3],    # covariance, variance of efficiency
    ],
    params=["specific_energy", "efficiency"],
    directions={
        "specific_energy": "more_is_better",
        "efficiency":      "more_is_better",
    },
)
drone = make_drone(bat)

r_ell = solve(drone, f, uncertainty=["worst_case"])
wc_ell = list(r_ell.worst_case.antichain.points)[0]["total_mass"]
print(f"Ellipsoid worst case: {wc_ell:.4f} kg  (penalty {wc_ell - nominal_mass:+.4f} kg)")

Ellipsoid worst case: 0.5527 kg  (penalty +0.0035 kg)


## Summary

| Set       | Worst-case mass | Penalty vs nominal |
|-----------|-----------------|--------------------|
| (nominal) | 0.5492 kg       | -                  |
| Box       | 0.5668 kg       | +0.0176 kg         |
| Ellipsoid | 0.5527 kg       | +0.0035 kg         |

The ellipsoid is the more honest model when you believe the two parameters are correlated, since it rejects the "both at the worst simultaneously" combination as implausible. The 2D conveniences `Disk(center, radius)` and `Circle(center, radius)` are special cases of `Ellipsoid`.
